# 02 · Campaign — assemble the pool + implement Layers 2 & 3

**Standard slot:** *design campaign.* **For Project 05 this means:** there is **nothing to generate** —
your "campaign" is *assembling* a labeled design pool and *building* the next two filter layers. You
generate the clearly-synthetic `EXAMPLE_DATA` pool (so the layers are developable with no GPU), then
implement and verify **Layer 2 (orthogonal)** and **Layer 3 (physics)** with unit-test asserts on
planted known-good / known-bad designs (D2).

Run `00_setup.ipynb` first in this session.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change!)

The engine needs no GPU, but its metrics come from upstream tools that move fast. Pin the upstream
repos and **verify they still exist** before relying on them. (`requests.head`; a non-200 means the
URL moved — update your pin and log it.)

In [ ]:
import requests

# Pinned upstream metric sources (verify; pin a commit/tag/version in your repo — these change):
#   Boltz      https://github.com/jwohlwend/boltz          (pin a release tag)
#   ColabFold  https://github.com/sokrypton/ColabFold      (pin a commit)
#   PyRosetta  https://www.pyrosetta.org                   (pin the release; academic license)
PINNED = {
    "Boltz (Layer 2 / affinity)":     "https://github.com/jwohlwend/boltz",
    "ColabFold (Layer 1/2 source)":   "https://github.com/sokrypton/ColabFold",
    "PyRosetta (Layer 3 physics)":    "https://www.pyrosetta.org",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:32s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:32s} {url}  ({e})")
print("\nNon-200 / error ⇒ the upstream moved; update the pin in env/requirements.txt and log it.")

## 1 · Build the labeled `EXAMPLE_DATA` pool

`scripts/make_example_pool.py` writes `results/pool.csv` (~200 mixed-type designs, deterministic
seed). **Every row is synthetic** (`design_id` prefixed `EXAMPLE_DATA_`), and a hidden `truth`
column records the *planted* label for the enrichment analysis in notebook 04. The pool deliberately
contains planted known-good, layer-specific known-bad, and ambiguous-near-cutoff designs — so
enrichment is imperfect and the discrimination problem is visible.

In [ ]:
import filtering_pipeline as fp
import make_example_pool as mep
import pandas as pd

df = mep.make_pool(n=200)
df.to_csv("results/pool.csv", index=False)
n_good = int((df["truth"] == "good").sum())
print(f"results/pool.csv: {len(df)} EXAMPLE_DATA designs ({n_good} planted-good, {len(df)-n_good} planted-bad)")
print("design types:", df["design_type"].value_counts().to_dict())
print("\nREMINDER: every number here is SYNTHETIC (EXAMPLE_DATA) — never report it as a real result.")
df.head(6)

## 2 · Implement & verify **Layer 2 — orthogonal agreement**

A design that *one* predictor (AF2) loves can still be wrong — predictors share blind spots and can be
overconfident on de novo sequences. Layer 2 requires a **second, independent** predictor (ESMFold or
Boltz, different inductive bias) to also reproduce the design: `scrmsd_orthogonal ≤ max_scrmsd`. The
planted `BAD_L2_orthogonal` design passes L1 but fails here — exactly the overconfidence case.

In [ ]:
planted = {r["design_id"]: r for r in mep.known_designs()}

def design_from_row(row):
    drop = {"truth", "note", "design_id"}
    fields = {k: v for k, v in row.items() if k not in drop}
    return fp.Design(design_id=row["design_id"], sequence="M", **fields)

cut_mono = fp.DEFAULT_CUTOFFS["monomer"]

good = design_from_row(planted["EXAMPLE_DATA_GOOD_monomer"])
assert fp.orthogonal_check(good) is True, "known-good should pass L2"

bad2 = design_from_row(planted["EXAMPLE_DATA_BAD_L2_orthogonal"])
assert fp.self_consistency(bad2, cut_mono) is True, "fixture is meant to PASS L1 (AF2 likes it)"
assert fp.orthogonal_check(bad2) is False, "orthogonal disagreement must FAIL L2"

print("Layer 2 verified:")
print(f"  GOOD_monomer         orthogonal scRMSD={good.scrmsd_orthogonal} -> pass")
print(f"  BAD_L2_orthogonal    orthogonal scRMSD={bad2.scrmsd_orthogonal} -> fail  notes={bad2.notes}")

## 3 · Implement & verify **Layer 3 — physics**

Self-consistency and orthogonal agreement say nothing about **solubility** or **interface strength**.
Layer 3 checks a CamSol-style `solubility` score and, for binders, interface energy (`rosetta_dG`,
REU) + shape complementarity (`sc`). Keep PyRosetta / FreeBindCraft / CamSol **behind the boundary** —
the layer thresholds numbers; the notebooks (on Colab) produce them. We verify two planted bads: an
aggregation-prone monomer and a weak-interface binder.

In [ ]:
cut_bind = fp.DEFAULT_CUTOFFS["binder"]

# Known-good monomer + binder pass L3:
for dt in ("monomer", "binder"):
    d = design_from_row(planted[f"EXAMPLE_DATA_GOOD_{dt}"])
    assert fp.physics_filter(d, fp.DEFAULT_CUTOFFS[dt]) is True, f"known-good {dt} should pass L3"

# Aggregation-prone monomer fails L3 on solubility:
bad_sol = design_from_row(planted["EXAMPLE_DATA_BAD_L3_solubility"])
assert fp.physics_filter(bad_sol, cut_mono) is False, "aggregation-prone monomer must fail L3"

# Weak-interface binder fails L3 on rosetta_dG / shape complementarity:
bad_int = design_from_row(planted["EXAMPLE_DATA_BAD_L3_interface"])
assert fp.physics_filter(bad_int, cut_bind) is False, "weak-interface binder must fail L3"

print("Layer 3 verified:")
print(f"  BAD_L3_solubility   solubility={bad_sol.solubility} -> fail  notes={bad_sol.notes}")
print(f"  BAD_L3_interface    rosetta_dG={bad_int.rosetta_dG}, sc={bad_int.shape_complementarity} -> fail")

## 4 · Run the test suite (the real D2 artifact)

The asserts above are spot checks; `scripts/test_filtering.py` is the durable contract. It imports
the **shared** `filtering_pipeline` and asserts every planted case across all four layers + the
orchestration. Run it here; it must exit 0. Re-run it after **any** cutoff change — a layer with no
failing planted case is untested.

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, "../scripts/test_filtering.py"],
                     capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print("STDERR:\n", res.stderr)
print("exit code:", res.returncode, "(0 = all tests passed)")

## D2 checklist
- [ ] `results/pool.csv`: labeled `EXAMPLE_DATA` pool (mixed types, planted + ambiguous), deterministic.
- [ ] Layer 2 (orthogonal) + Layer 3 (physics) implemented & verified on planted known-good/known-bad.
- [ ] `python scripts/test_filtering.py` exits 0 (per layer **and** per design type).
- [ ] Design log: every cutoff choice + seed + the test command + outcome, in `LOG.md`.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** engine on the pool.